In [1]:
import awkward as ak
import torch
import numpy as np
import vector
import tqdm
import os
import glob

from torch import nn
from torch.utils.data import Dataset
import torch.nn.functional as F

import sklearn
import sklearn.metrics
import matplotlib
import matplotlib.pyplot as plt

from omegaconf import OmegaConf
from omegaconf import DictConfig

from torch.utils.data import IterableDataset
from collections.abc import Sequence

import vbf_tagger.tools.data.general as g
from vbf_tagger.models.MLPClassifier import MLPClassifier
from torch.utils.data import DataLoader, TensorDataset

/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
vector.register_awkward()
def load_parquet(input_path: str, columns: list = None) -> ak.Array:
    """ Loads the contents of the .parquet file specified by the input_path
    Args:
        input_path : str
            The path to the .parquet file to be loaded.
        columns : list
            Names of the columns/branches to be loaded from the .parquet file
    Returns:
        input_data : ak.Array
            The data from the .parquet file
    """
    ret = ak.from_parquet(input_path, columns=columns)
    ret = ak.Array({k: ret[k] for k in ret.fields})
    return ret

def load_all_data(input_loc: str, n_files: int = None, columns: list = None) -> ak.Array:
    """Loads all .parquet files specified by the input. The input can be a list of input_paths, a directory where the files
    are located or a wildcard path.
    Args:
        input_loc : str
            Location of the .parquet files.
        n_files : int
            [default: None] Maximum number of input files to be loaded. By default all will be loaded.
        columns : list
            [default: None] Names of the columns/branches to be loaded from the .parquet file. By default all columns will
            be loaded
    Returns:
        input_data : ak.Array
            The concatenated data from all the loaded files
    """
    if n_files == -1:
        n_files = None
    if isinstance(input_loc, list):
        input_files = input_loc[:n_files]
    elif isinstance(input_loc, str):
        if os.path.isdir(input_loc):
            input_loc = os.path.expandvars(input_loc)
            input_files = glob.glob(os.path.join(input_loc, "*.parquet"))[:n_files]
        elif "*" in input_loc:
            input_files = glob.glob(input_loc)[:n_files]
        else:
            raise ValueError(f"Unexpected input_loc")
    else:
        raise ValueError(f"Unexpected input_loc")
    input_data = []
    for i, file_path in enumerate(input_files):
        print(f"[{i+1}/{len(input_files)}] Loading from {file_path}")
        try:
            input_data.append(load_parquet(file_path, columns=columns))
        except ValueError:
            print(f"{file_path} does not exist")
    if len(input_data) > 0:
        data = ak.concatenate(input_data)
        print("Input data loaded")
    else:
        raise ValueError(f"No files found in {input_loc}")
    return data

def initialize_p4(data):
    return ak.zip(
            {
                "pt": data.pt,
                "eta": data.eta,
                "phi": data.phi,
                "mass": data.mass,
            },
            with_name="Momentum4D"
        )

In [3]:
vbfhh = load_all_data("/home/norman/samples/hh_vbf/")

[1/7] Loading from /home/norman/samples/hh_vbf/hh_vbf_hbb_htt_kv1_k2v0_kl1_madgraph_events_0.parquet
[2/7] Loading from /home/norman/samples/hh_vbf/hh_vbf_hbb_htt_kv1_k2v2_kl1_madgraph_events_0.parquet
[3/7] Loading from /home/norman/samples/hh_vbf/hh_vbf_hbb_htt_kv1_k2v0_kl1_madgraph_events_1.parquet
[4/7] Loading from /home/norman/samples/hh_vbf/hh_vbf_hbb_htt_kv1_k2v1_kl2_madgraph_events_0.parquet
[5/7] Loading from /home/norman/samples/hh_vbf/hh_vbf_hbb_htt_kv1_k2v0_kl1_madgraph_events_2.parquet
[6/7] Loading from /home/norman/samples/hh_vbf/hh_vbf_hbb_htt_kv1_k2v1_kl1_madgraph_events_1.parquet
[7/7] Loading from /home/norman/samples/hh_vbf/hh_vbf_hbb_htt_kv1_k2v1_kl1_madgraph_events_0.parquet
Input data loaded


In [4]:
jets0 = initialize_p4(vbfhh.TrainingJet)
mask_valid = ak.num(jets0) > 3
isVBF = vbfhh.TrainingJet.isVBF[mask_valid]
jets = jets0[mask_valid]
jets_novec = vbfhh.TrainingJet[mask_valid]

In [5]:
pairs = ak.combinations(jets, 2, fields=["j1", "j2"])
pairs_isVBF = ak.combinations(isVBF, 2, fields=["j1", "j2"])
pairs_novec = ak.combinations(jets_novec, 2, fields=["j1", "j2"])

In [6]:
mjj = ak.flatten((pairs.j1 + pairs.j2).mass)
deta = ak.flatten(abs(pairs.j1.eta - pairs.j2.eta))
dphi = ak.flatten(abs(pairs.j1.phi - pairs.j2.phi))
ptjj = ak.flatten((pairs.j1 + pairs.j2).pt)
ejj = ak.flatten((pairs.j1 + pairs.j2).energy)

higher_pt_mask = abs(pairs.j1.pt) > abs(pairs.j2.pt)
min_pt_pair = ak.flatten(ak.where(higher_pt_mask, pairs.j2.pt, pairs.j1.pt))

# b-tag sums
btagDeepFlavB_sum = ak.flatten((pairs_novec.j1.btagDeepFlavB + pairs_novec.j2.btagDeepFlavB))

In [14]:
event_energy = ak.sum(jets.energy, axis=1)
event_energy_per_pair, _ = ak.broadcast_arrays(event_energy, pairs)

In [16]:
ak.flatten(event_energy_per_pair)

<Array [4.86e+03, 4.86e+03, ..., 2.66e+03, 2.66e+03] type='1272770 * float32'>

In [11]:
mjj

<Array [2.56e+03, 124, 446, 69.8, ..., 124, 118, 134] type='1272770 * float32'>

In [7]:
features_list0 = [
    mjj, ptjj, deta, dphi, ejj, min_pt_pair, btagDeepFlavB_sum
]

In [8]:
def build_flat_pair_features(data: ak.Array):
    # 1. Select events with > 3 jets
    jets = initialize_p4(data.TrainingJet)
    mask_valid = ak.num(jets) > 3
    jets = jets[mask_valid]
    isVBF = data.TrainingJet.isVBF[mask_valid]
    jets_novec = data.TrainingJet[mask_valid]

    # 2. Build all jet pairs
    pairs = ak.combinations(jets, 2, fields=["j1", "j2"])
    pairs_isVBF = ak.combinations(isVBF, 2, fields=["j1", "j2"])
    pairs_novec = ak.combinations(jets_novec, 2, fields=["j1", "j2"])

    # 3. Compute pair features
    mjj = ak.flatten((pairs.j1 + pairs.j2).mass)
    deta = ak.flatten(abs(pairs.j1.eta - pairs.j2.eta))
    dphi = ak.flatten(abs(pairs.j1.phi - pairs.j2.phi))
    ptjj = ak.flatten((pairs.j1 + pairs.j2).pt)
    dRjj = ak.flatten(pairs.j1.deltaR(pairs.j2))
    etaetajj = ak.flatten(pairs.j1.eta * pairs.j2.eta)
    denergyjj = ak.flatten(abs(pairs.j1.energy - pairs.j2.energy))
    ejj = ak.flatten((pairs.j1 + pairs.j2).energy)
    e_mjj = ejj / mjj
    higher_pt_mask = abs(pairs.j1.pt) > abs(pairs.j2.pt)
    min_pt_pair = ak.flatten(ak.where(higher_pt_mask, pairs.j2.pt, pairs.j1.pt))

    # b-tag sums
    btagDeepFlavB_sum = ak.flatten((pairs_novec.j1.btagDeepFlavB + pairs_novec.j2.btagDeepFlavB))
    btagDeepFlavCvB_sum = ak.flatten((pairs_novec.j1.btagDeepFlavCvB + pairs_novec.j2.btagDeepFlavCvB))
    btagDeepFlavCvL_sum = ak.flatten((pairs_novec.j1.btagDeepFlavCvL + pairs_novec.j2.btagDeepFlavCvL))
    btagDeepFlavQG_sum = ak.flatten((pairs_novec.j1.btagDeepFlavQG + pairs_novec.j2.btagDeepFlavQG))
    btagPNetB_sum = ak.flatten((pairs_novec.j1.btagPNetB + pairs_novec.j2.btagPNetB))
    btagPNetCvB_sum = ak.flatten((pairs_novec.j1.btagPNetCvB + pairs_novec.j2.btagPNetCvB))
    btagPNetCvL_sum = ak.flatten((pairs_novec.j1.btagPNetCvL + pairs_novec.j2.btagPNetCvL))
    btagPNetCvNotB_sum = ak.flatten((pairs_novec.j1.btagPNetCvNotB + pairs_novec.j2.btagPNetCvNotB))
    btagPNetQvG_sum = ak.flatten((pairs_novec.j1.btagPNetQvG + pairs_novec.j2.btagPNetQvG))
    btagPNetTauVJet_sum = ak.flatten((pairs_novec.j1.btagPNetTauVJet + pairs_novec.j2.btagPNetTauVJet))
    hhbtag_sum = ak.flatten((pairs_novec.j1.hhbtag + pairs_novec.j2.hhbtag))

    # # MET
    # PuppiMET_covXY = data.PuppiMET.covXY[mask_valid]
    # PuppiMET_pt = data.PuppiMET.pt[mask_valid]
    # PuppiMET_covXY_per_pair, _ = ak.broadcast_arrays(PuppiMET_covXY, pairs)
    # PuppiMET_pt_per_pair, _ = ak.broadcast_arrays(PuppiMET_pt, pairs)

    # # event-level vars
    # event_energy = ak.sum(jets.energy, axis=1)
    # event_pt = ak.sum(jets.pt, axis=1)
    # event_energy_per_pair, _ = ak.broadcast_arrays(event_energy, pairs)
    # event_pt_per_pair, _ = ak.broadcast_arrays(event_pt, pairs)

    # Label: both jets are VBF
    targets = ak.flatten((pairs_isVBF.j1 == 1) & (pairs_isVBF.j2 == 1))

    # 4. Package into a flat awkward array
    features = ak.Array({
        "mjj": mjj,
        "ptjj": ptjj,
        "deta": deta,
        "dphi": dphi,
        "btagDeepFlavB_sum": btagDeepFlavB_sum,
        "btagDeepFlavCvB_sum": btagDeepFlavCvB_sum,
        "btagDeepFlavCvL_sum": btagDeepFlavCvL_sum,
        "btagDeepFlavQG_sum": btagDeepFlavQG_sum,
        "btagPNetB_sum": btagPNetB_sum,
        "btagPNetCvB_sum": btagPNetCvB_sum,
        "btagPNetCvL_sum": btagPNetCvL_sum,
        "btagPNetCvNotB_sum": btagPNetCvNotB_sum,
        "btagPNetQvG_sum": btagPNetQvG_sum,
        "btagPNetTauVJet_sum": btagPNetTauVJet_sum,
        "hhbtag_sum": hhbtag_sum,
        "e_mjj": e_mjj,
        "dRjj": dRjj,
        "etaetajj": etaetajj,
        "denergyjj": denergyjj,
        "min_pt_pair": min_pt_pair,
        # "PuppiMET_covXY_per_pair": PuppiMET_covXY_per_pair,
        # "PuppiMET_pt_per_pair": PuppiMET_pt_per_pair,   
        # "event_energy_per_pair": event_energy_per_pair,
        # "event_pt_per_pair": event_pt_per_pair,
    })

    # 5. Flatten across all events
    X = ak.to_numpy(ak.values_astype(features, np.float32))
    X = np.stack([X[name] for name in X.dtype.names], axis=1)
    y = ak.to_numpy(ak.values_astype(targets, np.int8))
    y = np.array(y)

    return X, y

In [9]:
X, y = build_flat_pair_features(vbfhh)

In [26]:
X

array([[ 2.55671021e+03,  2.40848480e+02,  5.34643555e+00, ...,
        -5.54150438e+00,  1.99340710e+03,  1.02853569e+02],
       [ 1.23636055e+02,  3.44476196e+02,  1.08886719e-01, ...,
         1.82507730e+00,  5.35703064e+02,  5.86818733e+01],
       [ 4.45938904e+02,  3.39852783e+02,  2.52563477e+00, ...,
         5.53051615e+00,  7.68951660e+02,  5.56751823e+01],
       ...,
       [ 1.23950676e+02,  4.44309006e+01,  1.22283936e+00, ...,
        -2.91033804e-01,  6.83046036e+01,  3.68236885e+01],
       [ 1.18350845e+02,  6.45303802e+01,  1.45385742e+00, ...,
         2.11577129e+00,  6.89063110e+01,  3.32183952e+01],
       [ 1.33671555e+02,  5.18502998e+01,  2.67669678e+00, ...,
        -7.61602998e-01,  1.37210907e+02,  3.32183952e+01]],
      shape=(1272770, 20), dtype=float32)

In [27]:
y

array([0, 0, 0, ..., 0, 0, 0], shape=(1272770,), dtype=int8)

In [28]:
X2 = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-6)

In [29]:
X2

array([[ 4.363053  ,  0.8162654 ,  1.5538245 , ..., -1.1970571 ,
         2.3515882 ,  1.4363885 ],
       [-0.5158713 ,  1.6684922 , -1.3439507 , ...,  0.46802923,
         0.04554241,  0.32573998],
       [ 0.1304268 ,  1.6304696 , -0.00683807, ...,  1.3055787 ,
         0.414535  ,  0.25014007],
       ...,
       [-0.5152404 , -0.7990584 , -0.72763485, ..., -0.01028051,
        -0.6938685 , -0.22385994],
       [-0.5264694 , -0.63376176, -0.5998196 , ...,  0.5337355 ,
        -0.69291663, -0.31451106],
       [-0.4957476 , -0.7380418 ,  0.07673991, ..., -0.11664441,
        -0.58486074, -0.31451106]], shape=(1272770, 20), dtype=float32)

In [ ]:
n_pos = np.sum(y == 1)
n_neg = np.sum(y == 0)
posw = n_neg / n_pos
print(f" Class balance: {n_pos} positives, {n_neg} negatives → pos_weight={posw:.2f}")

In [23]:
class FlatPairDataModule(LightningDataModule):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.pos_weight = None

    def setup(self, stage=None):
        data = ak.from_parquet(self.cfg.dataset.train_files)
        X, y = build_flat_pair_features(data)

        # Compute pos_weight = (#neg / #pos)
        n_pos = np.sum(y == 1)
        n_neg = np.sum(y == 0)
        self.pos_weight = n_neg / n_pos
        print(f" Class balance: {n_pos} positives, {n_neg} negatives → pos_weight={self.pos_weight:.2f}")

        # Convert to tensors
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)

        self.train_data = TensorDataset(X, y)

    def train_dataloader(self):
        return DataLoader(
            self.train_data,
            batch_size=self.cfg.training.dataloader.batch_size,
            shuffle=True
        )

NameError: name 'LightningDataModule' is not defined